In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "cointegrated/rut5-base-multitask"

INPUT_CSV_PATH = "/content/eval_source_target_sentences.csv"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/eval_result_noncontr.сsv"
TEMP_OUTPUT_CSV_PATH = "/content/drive/MyDrive/eval_result_noncontr_tmp.сsv"

MAX_NEW_TOKENS = 64
BATCH_SIZE = 16
NUM_BEAMS = 4
RESUME = True
SAVE_EVERY_N_BATCHES = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()

def build_prompt(sentence: str) -> str:
    return f"simplify | {sentence}"

def simplify_batch(sentences):
    prompts = [build_prompt(s) for s in sentences]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    results = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    results = [r.strip().replace("\n", " ") for r in results]
    return results

def safe_save_csv(df, output_path, temp_path=None):
    if temp_path is None:
        temp_path = output_path + ".tmp"
    df.to_csv(temp_path, index=False, sep="\t")
    os.replace(temp_path, output_path)

def simplify_csv(input_csv_path, output_csv_path, batch_size=BATCH_SIZE, resume=RESUME):
    df = pd.read_csv(input_csv_path)

    if "source" not in df.columns:
        raise ValueError(f"Во входном файле должна быть колонка 'source'. Найдены: {list(df.columns)}")

    sources = df["source"].fillna("").astype(str).tolist()
    total_rows = len(sources)

    start_idx = 0
    results = []

    if resume and os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path, sep="\t")

        if not {"source", "result"}.issubset(existing_df.columns):
            raise ValueError(
                f"В checkpoint-файле должны быть колонки 'source' и 'result'. "
                f"Найдены: {list(existing_df.columns)}"
            )

        existing_sources = existing_df["source"].fillna("").astype(str).tolist()
        existing_results = existing_df["result"].fillna("").astype(str).tolist()

        prefix_len = len(existing_sources)
        if prefix_len > total_rows:
            raise ValueError("Checkpoint длиннее входного файла.")

        if existing_sources != sources[:prefix_len]:
            raise ValueError("Checkpoint не совпадает с началом входного файла.")

        results = existing_results
        start_idx = prefix_len
        print(f"Resume enabled: найден checkpoint на {start_idx} строк(ах) из {total_rows}.")
    else:
        print("Checkpoint не найден. Начинаем с нуля.")

    progress_bar = tqdm(
        range(start_idx, total_rows, batch_size),
        desc="Simplifying",
        unit="batch"
    )

    for batch_num, i in enumerate(progress_bar, start=1):
        batch_sources = sources[i:i + batch_size]
        batch_results = simplify_batch(batch_sources)
        results.extend(batch_results)

        progress_bar.set_postfix({
            "processed_rows": len(results),
            "total_rows": total_rows
        })

        if batch_num % SAVE_EVERY_N_BATCHES == 0:
            out_df = pd.DataFrame({
                "source": sources[:len(results)],
                "result": results
            })
            safe_save_csv(out_df, output_csv_path, TEMP_OUTPUT_CSV_PATH)

    out_df = pd.DataFrame({
        "source": sources,
        "result": results
    })

    safe_save_csv(out_df, output_csv_path, TEMP_OUTPUT_CSV_PATH)
    return out_df

result_df = simplify_csv(
    INPUT_CSV_PATH,
    OUTPUT_CSV_PATH,
    batch_size=BATCH_SIZE,
    resume=RESUME
)

print(f"Saved to: {OUTPUT_CSV_PATH}")
print(result_df.head())

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Checkpoint не найден. Начинаем с нуля.


Simplifying:   0%|          | 0/168 [00:00<?, ?batch/s]

Saved to: /content/drive/MyDrive/eval_result_noncontr.сsv
                                              source  \
0  Или дитя ваше спросит вас: «Что ждет меня в за...   
1  Но он уже пригляделся, так что мог различать в...   
2  Свобода, свободный ум и наука заведут их в так...   
3  К третьему и окончательному штурму плевненской...   
4             Стройный, высокий, румянец во всю щеку   

                                              result  
0  Детя вас спросит: «Что ждет меня в загробной ж...  
1  Он уже пригляделся, так что мог различать всю ...  
2  Свобода, свободный ум и наука заведут их в так...  
3  К окончательному штурму плевненской твердыни г...  
4                               Румянец во всю щеку.  


In [ ]:
import os
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "cointegrated/rut5-base-multitask"

INPUT_CSV_PATH = "/content/eval_source_target_sentences.csv"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/eval_result_token.сsv"
TEMP_OUTPUT_CSV_PATH = "/content/drive/MyDrive/eval_result_token_tmp.сsv"

MAX_NEW_TOKENS = 64
BATCH_SIZE = 16
NUM_BEAMS = 5
RESUME = True
SAVE_EVERY_N_BATCHES = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE).eval()

def build_prompt(sentence: str) -> str:
    return f"simplify <level A1-A2> | {sentence}"

def simplify_batch(sentences):
    prompts = [build_prompt(s) for s in sentences]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    ).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )

    results = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    results = [r.strip().replace("\n", " ") for r in results]
    return results

def safe_save_csv(df, output_path, temp_path=None):
    if temp_path is None:
        temp_path = output_path + ".tmp"
    df.to_csv(temp_path, index=False, sep="\t")
    os.replace(temp_path, output_path)

def simplify_csv(input_csv_path, output_csv_path, batch_size=BATCH_SIZE, resume=RESUME):
    df = pd.read_csv(input_csv_path)

    if "source" not in df.columns:
        raise ValueError(f"Во входном файле должна быть колонка 'source'. Найдены: {list(df.columns)}")

    sources = df["source"].fillna("").astype(str).tolist()[:10]
    total_rows = len(sources)

    start_idx = 0
    results = []

    if resume and os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path, sep="\t")

        if not {"source", "result"}.issubset(existing_df.columns):
            raise ValueError(
                f"В checkpoint-файле должны быть колонки 'source' и 'result'. "
                f"Найдены: {list(existing_df.columns)}"
            )

        existing_sources = existing_df["source"].fillna("").astype(str).tolist()
        existing_results = existing_df["result"].fillna("").astype(str).tolist()

        prefix_len = len(existing_sources)
        if prefix_len > total_rows:
            raise ValueError("Checkpoint длиннее входного файла.")

        if existing_sources != sources[:prefix_len]:
            raise ValueError("Checkpoint не совпадает с началом входного файла.")

        results = existing_results
        start_idx = prefix_len
        print(f"Resume enabled: найден checkpoint на {start_idx} строк(ах) из {total_rows}.")
    else:
        print("Checkpoint не найден. Начинаем с нуля.")

    progress_bar = tqdm(
        range(start_idx, total_rows, batch_size),
        desc="Simplifying",
        unit="batch"
    )

    for batch_num, i in enumerate(progress_bar, start=1):
        batch_sources = sources[i:i + batch_size]
        batch_results = simplify_batch(batch_sources)
        results.extend(batch_results)

        progress_bar.set_postfix({
            "processed_rows": len(results),
            "total_rows": total_rows
        })

        if batch_num % SAVE_EVERY_N_BATCHES == 0:
            out_df = pd.DataFrame({
                "source": sources[:len(results)],
                "result": results
            })
            safe_save_csv(out_df, output_csv_path, TEMP_OUTPUT_CSV_PATH)

    out_df = pd.DataFrame({
        "source": sources,
        "result": results
    })

    safe_save_csv(out_df, output_csv_path, TEMP_OUTPUT_CSV_PATH)
    return out_df


result_df = simplify_csv(
    INPUT_CSV_PATH,
    OUTPUT_CSV_PATH,
    batch_size=BATCH_SIZE,
    resume=RESUME
)

print(f"Saved to: {OUTPUT_CSV_PATH}")
print(result_df.head())

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Checkpoint не найден. Начинаем с нуля.


Simplifying:   0%|          | 0/1 [00:00<?, ?batch/s]

Saved to: /content/drive/MyDrive/eval_result_token.сsv
                                              source  \
0  Или дитя ваше спросит вас: «Что ждет меня в за...   
1  Но он уже пригляделся, так что мог различать в...   
2  Свобода, свободный ум и наука заведут их в так...   
3  К третьему и окончательному штурму плевненской...   
4             Стройный, высокий, румянец во всю щеку   

                                              result  
0  Детя ваше спросит тебя: «Что ждет тебя в загро...  
1  Он пригляделся, так что мог различать всю пост...  
2                                      <level A1-A2>  
3                                      <level A1-A2>  
4  <level A1-A2> Стройный, высокий, румянец во вс...  


In [ ]:
import os
import re
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Literal
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    LogitsProcessorList,
    BitsAndBytesConfig,
)


@dataclass
class BeamSearchConfig:
    level_group: Literal["A", "B", "C"]
    condition_lambda: float = 0.2
    topk: int = 5
    num_beams: int = 3
    max_new_tokens: int = 40
    do_sample: bool = False
    wait: int = 10
    load_base_in_4bit: bool = True


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_save_csv(df: pd.DataFrame, output_path: str, temp_path: str | None = None, sep: str = "\t"):
    if temp_path is None:
        temp_path = output_path + ".tmp"
    df.to_csv(temp_path, index=False, sep=sep)
    os.replace(temp_path, output_path)


def build_prompt(sentence: str, level_group: str) -> str:
    return f"<lvl_{level_group}> Упрости: {sentence}"


def clean_result(text: str) -> str:
    text = text.strip().replace("\n", " ")

    text = re.sub(r"<lvl_[ABC]>", "", text, flags=re.IGNORECASE).strip()

    prefixes = [
        "Упрости:",
        "Ответ:",
        "Упрощённый вариант:",
        "Упрощенный вариант:",
        "В ответе верни только упрощённое предложение одной строкой.",
        "В ответе верни только упрощенное предложение одной строкой.",
        "Answer on russian:",
        "model",
    ]

    changed = True
    while changed:
        changed = False
        for p in prefixes:
            if text.lower().startswith(p.lower()):
                text = text[len(p):].strip()
                changed = True

    for marker in ["```", "{", "}", '", "', "CreateTagHelper", "JSON", "json"]:
        pos = text.find(marker)
        if pos > 0:
            text = text[:pos].strip()

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[0]

    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"^(model\s+)+", "", text).strip()

    return text


def is_bad_output(text: str) -> bool:
    if not text:
        return True
    if len(text) < 3:
        return True
    bad_markers = [
        "CreateTagHelper",
        "```",
        "<tool_call>",
        "Answer on russian",
        "model",
        '{"',
    ]
    if any(marker.lower() in text.lower() for marker in bad_markers):
        return True

    latin = sum(ch.isascii() and ch.isalpha() for ch in text)
    cyr = sum("а" <= ch.lower() <= "я" or ch.lower() == "ё" for ch in text)
    if latin > cyr and latin > 8:
        return True

    return False


class RussianCEFRScorer:
    GROUP_MAP = {
        "A": [0, 1],
        "B": [2, 3],
        "C": [4, 5],
    }

    def __init__(self, model, tokenizer, level_group: str, device: str = "cuda"):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.indices = self.GROUP_MAP[level_group]
        self.device = device

    @torch.no_grad()
    def score(self, texts: List[str]) -> torch.Tensor:
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=256,
        ).to(self.device)

        outputs = self.model(**inputs)
        logits = outputs.logits
        log_probs = F.log_softmax(logits, dim=-1)
        return torch.logsumexp(log_probs[:, self.indices], dim=-1)


class LevelWeighter:
    def __init__(self, tokenizer, prompt_len, scorer, cfg):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len
        self.scorer = scorer
        self.cfg = cfg

    def __call__(self, input_ids, scores):
        generated_ids = input_ids[:, self.prompt_len:]

        if generated_ids.shape[-1] < self.cfg.wait:
            return scores

        top_scores, top_ids = torch.topk(scores, self.cfg.topk, dim=1)
        bsz, k = top_ids.shape

        prefix_rep = generated_ids.repeat_interleave(k, dim=0)
        next_tok = top_ids.reshape(-1, 1)
        candidate_seq = torch.cat([prefix_rep, next_tok], dim=1)

        candidate_texts = self.tokenizer.batch_decode(candidate_seq, skip_special_tokens=True)
        cond = self.scorer.score(candidate_texts).view(bsz, k).to(scores.device)

        new_top = top_scores + self.cfg.condition_lambda * cond
        out = scores.clone()
        out.scatter_(1, top_ids, new_top)
        return out

-
class ConditionalBeamSearch:
    def __init__(self, cfg: BeamSearchConfig):
        if not torch.cuda.is_available():
            raise RuntimeError("Enable GPU runtime in Colab.")

        self.cfg = cfg
        self.device = torch.device("cuda")

        self.model, self.tokenizer = self._load_base_model()
        self.scorer = self._load_scorer()

    def _load_base_model(self):
        base_model = "Vikhrmodels/Vikhr-Gemma-2B-instruct"

        tokenizer = AutoTokenizer.from_pretrained(base_model)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        if self.cfg.load_base_in_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )
            model = AutoModelForCausalLM.from_pretrained(
                base_model,
                quantization_config=bnb_config,
                device_map={"": 0},
            ).eval()
        else:
            model = AutoModelForCausalLM.from_pretrained(
                base_model,
                torch_dtype=torch.float16,
            ).to(self.device).eval()

        return model, tokenizer

    def _load_scorer(self):
        cond_model_name = "tareknaous/readabert-ru"
        tokenizer = AutoTokenizer.from_pretrained(cond_model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            cond_model_name
        ).to("cuda").half().eval()

        return RussianCEFRScorer(model, tokenizer, self.cfg.level_group, device="cuda")

    def _encode_single(self, prompt: str):
        return self.tokenizer(
            prompt,
            truncation=True,
            padding=False,
            return_tensors="pt",
            max_length=256,
        )

    def generate_one(self, prompt: str) -> str:
        enc = self._encode_single(prompt)
        input_ids = enc["input_ids"].to("cuda")
        attention_mask = enc.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to("cuda")

        prompt_len = input_ids.shape[-1]
        processor = LevelWeighter(self.tokenizer, prompt_len, self.scorer, self.cfg)

        out = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=self.cfg.max_new_tokens,
            num_beams=self.cfg.num_beams,
            do_sample=self.cfg.do_sample,
            logits_processor=LogitsProcessorList([processor]),
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
        )

        gen_ids = out[0, prompt_len:]
        text = self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        text = clean_result(text)

        if is_bad_output(text):
            text = self.generate_one_plain(prompt)

        return text

    def generate_one_plain(self, prompt: str) -> str:
        enc = self._encode_single(prompt)
        input_ids = enc["input_ids"].to("cuda")
        attention_mask = enc.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to("cuda")

        prompt_len = input_ids.shape[-1]

        out = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=self.cfg.max_new_tokens,
            num_beams=1,
            do_sample=False,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
            repetition_penalty=1.1,
            no_repeat_ngram_size=3,
        )

        gen_ids = out[0, prompt_len:]
        text = self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        return clean_result(text)

    def generate(self, prompts: List[str]) -> List[str]:
        results = []
        for prompt in prompts:
            results.append(self.generate_one(prompt))
        return results

def simplify_csv(
    input_csv_path: str,
    output_csv_path: str,
    generator: ConditionalBeamSearch,
    level_group: str,
    batch_size: int = 8,
    resume: bool = True,
    save_every_n_batches: int = 1,
    temp_output_path: str | None = None,
    sep: str = "\t",
) -> str:
    df = pd.read_csv(input_csv_path)

    if "source" not in df.columns:
        raise ValueError(f"Input file must contain column 'source'. Found: {list(df.columns)}")

    sources = df["source"].fillna("").astype(str).tolist()
    total_rows = len(sources)

    start_idx = 0
    results: List[str] = []

    if resume and os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path, sep=sep)

        if not {"source", "result", "level"}.issubset(existing_df.columns):
            raise ValueError(
                f"Checkpoint must contain columns 'source', 'result', 'level'. Found: {list(existing_df.columns)}"
            )

        existing_sources = existing_df["source"].fillna("").astype(str).tolist()
        existing_results = existing_df["result"].fillna("").astype(str).tolist()

        prefix_len = len(existing_sources)
        if prefix_len > total_rows:
            raise ValueError("Checkpoint is longer than input file.")

        if existing_sources != sources[:prefix_len]:
            raise ValueError("Checkpoint does not match the beginning of input file.")

        results = existing_results
        start_idx = prefix_len
        print(f"Resume enabled: found checkpoint for {start_idx} rows out of {total_rows}.")
    else:
        print("Checkpoint not found. Starting from scratch.")

    progress_bar = tqdm(
        range(start_idx, total_rows, batch_size),
        desc="Simplifying",
        unit="batch"
    )

    for batch_num, i in enumerate(progress_bar, start=1):
        batch_sources = sources[i:i + batch_size]
        prompts = [build_prompt(s, level_group) for s in batch_sources]
        batch_out = generator.generate(prompts)
        results.extend(batch_out)

        progress_bar.set_postfix({
            "processed_rows": len(results),
            "total_rows": total_rows
        })

        if batch_num % save_every_n_batches == 0:
            out_df = pd.DataFrame({
                "source": sources[:len(results)],
                "result": results,
                "level": [level_group] * len(results),
            })
            safe_save_csv(out_df, output_csv_path, temp_output_path, sep=sep)

    out_df = pd.DataFrame({
        "source": sources,
        "result": results,
        "level": [level_group] * len(sources),
    })
    safe_save_csv(out_df, output_csv_path, temp_output_path, sep=sep)

    return output_csv_path


cleanup()

cfg = BeamSearchConfig(
    level_group="A",
    condition_lambda=0.2,
    topk=5,
    num_beams=3,
    max_new_tokens=40,
    do_sample=False,
    wait=10,
    load_base_in_4bit=True,
)

gen = ConditionalBeamSearch(cfg)

INPUT_CSV = "/content/eval_source_target_sentences.csv"
OUTPUT_CSV = "/content/drive/MyDrive/eval_result_contr_vikhr.tsv"
TEMP_OUTPUT_CSV = "/content/drive/MyDrive/eval_result_contr_vikhr.tsv"

out_path = simplify_csv(
    input_csv_path=INPUT_CSV,
    output_csv_path=OUTPUT_CSV,
    generator=gen,
    level_group=cfg.level_group,
    batch_size=8,
    resume=True,
    save_every_n_batches=1,
    temp_output_path=TEMP_OUTPUT_CSV,
    sep="\t",
)

print("Saved:", out_path)
print(pd.read_csv(out_path, sep="\t").head())

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.20 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.53 GiB is free. Process 688 has 12.77 GiB memory in use. Including non-PyTorch memory, this process has 268.00 MiB memory in use. Of the allocated memory 164.01 MiB is allocated by PyTorch, and 1.99 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import os
import gc
import pandas as pd
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Literal, Optional
from tqdm.auto import tqdm

from transformers import (
    AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
    BitsAndBytesConfig,
)

@dataclass
class BeamSearchConfig:
    level_group: Literal["A", "B", "C"]
    condition_lambda: float = 0.8
    topk: int = 10
    num_beams: int = 2
    max_new_tokens: int = 48
    do_sample: bool = False
    wait: int = 3
    load_base_in_4bit: bool = True


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_save_csv(df: pd.DataFrame, output_path: str, temp_path: Optional[str] = None, sep: str = "\t"):
    if temp_path is None:
        temp_path = output_path + ".tmp"
    df.to_csv(temp_path, index=False, sep=sep)
    os.replace(temp_path, output_path)


class RussianCEFRScorer:
    GROUP_MAP = {
        "A": [0, 1],
        "B": [2, 3],
        "C": [4, 5],
    }

    def __init__(self, model, tokenizer, level_group: str, device: str = "cuda"):
        self.model = model.eval()
        self.tokenizer = tokenizer
        self.indices = self.GROUP_MAP[level_group]
        self.device = device

    @torch.no_grad()
    def score(self, texts: List[str]) -> torch.Tensor:
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=256,
        ).to(self.device)

        outputs = self.model(**inputs)
        logits = outputs.logits
        log_probs = F.log_softmax(logits, dim=-1)
        return torch.logsumexp(log_probs[:, self.indices], dim=-1)


class LevelWeighter(LogitsProcessor):
    def __init__(self, tokenizer, scorer, cfg):
        super().__init__()
        self.tokenizer = tokenizer
        self.scorer = scorer
        self.cfg = cfg
        self.generated_so_far = 0

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        self.generated_so_far += 1

        if self.generated_so_far < self.cfg.wait:
            return scores


        top_scores, top_ids = torch.topk(scores, self.cfg.topk, dim=1)
        bsz, k = top_ids.shape

        return scores

class ConditionalBeamSearch:
    def __init__(self, cfg: BeamSearchConfig):
        if not torch.cuda.is_available():
            raise RuntimeError("Enable GPU runtime in Colab.")

        self.cfg = cfg
        self.device = torch.device("cuda")

        self.model, self.tokenizer = self._load_base_model()
        self.scorer = self._load_scorer()

    def _load_base_model(self):
        base_model = "Vikhrmodels/Vikhr-Gemma-2B-instruct"
        # base_model = "IlyaGusev/saiga_llama3_8b"
        # base_model = "Vikhrmodels/Vikhr-7B-instruct"

        tokenizer = AutoTokenizer.from_pretrained(base_model)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        if self.cfg.load_base_in_4bit:
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
                bnb_4bit_compute_dtype=torch.float16,
            )

            from transformers import AutoModelForCausalLM
            model = AutoModelForCausalLM.from_pretrained(
                base_model,
                quantization_config=bnb_config,
                device_map={"": 0},
                torch_dtype=torch.float16,
            ).eval()
        else:
            from transformers import AutoModelForCausalLM
            model = AutoModelForCausalLM.from_pretrained(
                base_model,
                torch_dtype=torch.float16,
                device_map="auto",
            ).eval()

        return model, tokenizer

    def _load_scorer(self):
        cond_model_name = "tareknaous/readabert-ru"

        tokenizer = AutoTokenizer.from_pretrained(cond_model_name)
        model = AutoModelForSequenceClassification.from_pretrained(
            cond_model_name
        ).to("cuda").half().eval()

        return RussianCEFRScorer(model, tokenizer, self.cfg.level_group, device="cuda")

    def _encode(self, prompts: List[str]):
        prompt_texts = [
            f"Упрости предложение, сохранив его смысл:\n{p}"
            for p in prompts
        ]

        return self.tokenizer(
            prompt_texts,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=256,
        )

    def generate(self, prompts: List[str]) -> List[str]:
        enc = self._encode(prompts)

        input_ids = enc["input_ids"].to(self.device)
        attention_mask = enc.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=self.cfg.max_new_tokens,
                num_beams=self.cfg.num_beams,
                do_sample=self.cfg.do_sample,
                temperature=0.7 if self.cfg.do_sample else 1.0,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,
            )

        results = []
        for i, output in enumerate(outputs):
            prompt_length = input_ids.shape[1]
            generated_ids = output[prompt_length:]
            text = self.tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

            for prefix in ("Ответ:", "Упрости предложение на русском языке:"):
                if text.startswith(prefix):
                    text = text[len(prefix):].strip()

            results.append(text)

        return results

def simplify_csv(
    input_csv_path: str,
    output_csv_path: str,
    generator: ConditionalBeamSearch,
    level_group: str,
    batch_size: int = 8,
    prompt_prefix: str = "",
    resume: bool = True,
    save_every_n_batches: int = 1,
    temp_output_path: Optional[str] = None,
    sep: str = "\t",
    max_rows: Optional[int] = None,
) -> str:
    df = pd.read_csv(input_csv_path)

    if "source" not in df.columns:
        raise ValueError(f"Input CSV must contain column 'source'. Found columns: {list(df.columns)}")

    sources = df["source"].fillna("").astype(str).tolist()

    if max_rows is not None:
        sources = sources[:max_rows]

    total_rows = len(sources)

    start_idx = 0
    results: List[str] = []

    if resume and os.path.exists(output_csv_path):
        existing_df = pd.read_csv(output_csv_path, sep=sep)

        if not {"source", "result", "level"}.issubset(existing_df.columns):
            raise ValueError(
                f"Checkpoint must contain columns 'source', 'result', 'level'. "
                f"Found: {list(existing_df.columns)}"
            )

        existing_sources = existing_df["source"].fillna("").astype(str).tolist()
        existing_results = existing_df["result"].fillna("").astype(str).tolist()

        prefix_len = len(existing_sources)
        if prefix_len > total_rows:
            raise ValueError("Checkpoint is longer than input file.")

        current_prefix = sources[:prefix_len]
        if existing_sources != current_prefix:
            print("Warning: Checkpoint does not match the beginning of the input file. Starting from scratch.")
            results = []
            start_idx = 0
        else:
            results = existing_results
            start_idx = prefix_len
            print(f"Resume enabled: found checkpoint for {start_idx} rows out of {total_rows}.")
    else:
        print("Checkpoint not found. Starting from scratch.")

    progress_bar = tqdm(
        total=total_rows,
        initial=start_idx,
        desc="Simplifying",
        unit="rows"
    )

    for batch_start in range(start_idx, total_rows, batch_size):
        batch_end = min(batch_start + batch_size, total_rows)
        batch = sources[batch_start:batch_end]

        prompts = [prompt_prefix + s for s in batch]

        try:
            batch_out = generator.generate(prompts)
            results.extend(batch_out)

            progress_bar.update(len(batch))
            progress_bar.set_postfix({
                "processed": len(results),
                "total": total_rows
            })

            if (batch_end - start_idx) % (batch_size * save_every_n_batches) == 0 or batch_end == total_rows:
                out_df = pd.DataFrame({
                    "source": sources[:len(results)],
                    "result": results,
                    "level": [level_group] * len(results),
                })
                safe_save_csv(out_df, output_csv_path, temp_output_path, sep=sep)

        except Exception as e:
            print(f"Error processing batch {batch_start}-{batch_end}: {e}")
            out_df = pd.DataFrame({
                "source": sources[:len(results)],
                "result": results,
                "level": [level_group] * len(results),
            })
            safe_save_csv(out_df, output_csv_path, temp_output_path, sep=sep)
            raise e

    progress_bar.close()

    out_df = pd.DataFrame({
        "source": sources,
        "result": results,
        "level": [level_group] * len(sources),
    })

    safe_save_csv(out_df, output_csv_path, temp_output_path, sep=sep)
    return output_csv_path


if __name__ == "__main__":
    cleanup()

    cfg = BeamSearchConfig(
        level_group="A",
        condition_lambda=0.8,
        topk=10,
        num_beams=2,
        max_new_tokens=48,
        do_sample=False,
        load_base_in_4bit=True,
    )

    print("Initializing model...")
    gen = ConditionalBeamSearch(cfg)

    INPUT_CSV = "eval_source_target_sentences.csv"
    OUTPUT_CSV = "eval_result_contr_llama.tsv"

    if os.path.exists(INPUT_CSV):
        out_path = simplify_csv(
            input_csv_path=INPUT_CSV,
            output_csv_path=OUTPUT_CSV,
            generator=gen,
            level_group=cfg.level_group,
            batch_size=4,
            prompt_prefix="",
            resume=True,
            save_every_n_batches=1,
            sep="\t",
            max_rows=None,
        )

        print("Saved:", out_path)
        if os.path.exists(out_path):
            print(pd.read_csv(out_path, sep="\t").head())
    else:
        print(f"Input file not found: {INPUT_CSV}")

Initializing model...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Checkpoint not found. Starting from scratch.


Simplifying:   0%|          | 0/10 [00:00<?, ?rows/s]

Saved: eval_result_contr_llama.tsv
                                              source  \
0  Или дитя ваше спросит вас: «Что ждет меня в за...   
1  Но он уже пригляделся, так что мог различать в...   
2  Свобода, свободный ум и наука заведут их в так...   
3  К третьему и окончательному штурму плевненской...   
4             Стройный, высокий, румянец во всю щеку   

                                              result level  
0  или «Что будет с нами после смерти?»\nИли «Что...     A  
1  model\nДля упрощения предложения, сохраняя при...     A  
2  model\nДля упрощения предложения, сохраняя его...     A  
3  model\nДля упрощения предложения, сохраняя при...     A  
4  .\nmodel\nДля упрощения предложения, сохраняя ...     A  
